# Egress personas — a look at one sampled population

A thin driver. Everything here calls the package; no logic lives in this notebook,
so nothing can drift between what you see here and what `personas build` writes.

Run `uv sync` first, then this from the repository root.


In [ ]:
from pathlib import Path

from egress_personas.tables import load_dir
from egress_personas.validate import problems
from egress_personas.sample import sample
from egress_personas.report import markdown

tables = load_dir(Path('data'))
probs = problems(tables)
print(f"snapshot {tables.snapshot['content_hash']}")
for p in probs:
    print(' ', p)
assert not probs, 'fix the Sheet before sampling from it'


## Sample a population


In [ ]:
pop = sample(tables, 'night_fire12', seed=1234)
print(pop.run_id)
print(f'{len(pop.people)} residents, {len(pop.households)} households')
print(f'household-size tilt theta = {pop.theta:.4f}')


## Targets against what was drawn

`z` is how far the drawn share sits from its target in standard errors. A row
outside its tolerance but inside about three standard errors is a small building,
not a biased sampler.


In [ ]:
for c in pop.conformance:
    if c['within'] is None:
        continue
    mark = 'ok' if c['within'] else ('noise' if c.get('noise') else 'OFF')
    z = '' if c.get('z') is None else f"z={c['z']:+.2f}"
    print(f"{c['dimension']+'.'+c['category']:42} "
          f"want {c['target']:<7} got {c['realized']:<7} {mark:<6} {z}")


## Who is in the building


In [ ]:
from collections import Counter

for label, values in [
    ('type', [p.body['sim_agent_type'] for p in pop.people]),
    ('mobility', [p.body['mobility'] for p in pop.people]),
    ('age band', [p.identity['age_band'] for p in pop.people]),
    ('household size', [len(h.members) for h in pop.households]),
]:
    print(label)
    for k, n in sorted(Counter(values).items(), key=lambda kv: -kv[1]):
        print(f'   {str(k):22} {n:4}  ' + '#' * n)
    print()


## The social graph

The last number is the one to watch: if nobody is isolated, the model has lost the
resident who has just moved in and knows nobody.


In [ ]:
from egress_personas.ties import degree_summary

for s in pop.tie_stats:
    print(f"{s['rule']} {s['kind']:12} {s['scope']:16} stated {s['rate']}"
          f" ({s['rate_kind']}) -> {s['realized_degree']} per person")
print()
for kind, d in sorted(degree_summary(pop.people).items()):
    print(f"{kind:12} mean {d['mean_over_all']:.3f}  most {int(d['max'])}"
          f"  any {d['share_with_any']:.1%}")
alone = [p for p in pop.people if not p.ties]
print(f'\n{len(alone)} resident(s) know nobody:')
for p in alone:
    print(f"   {p.id} {p.situation['unit']} tenure={p.identity['tenure_years']}y")


## One persona, in full

The card is what would condition a model. The table below it is where every number
came from — which row, which distribution, which paper.


In [ ]:
who = next(p for p in pop.people if p.case_id == 'C01')
print(who.narrative)
print()
for path, tr in sorted(pop.traces[who.key].items()):
    if tr.inherited:
        print(f'{path:38} inherited from the simulation')
        continue
    chain = ' then '.join(f'{s.rule} {s.mechanism}' for s in tr.steps if not s.shadowed)
    src = sorted({x for s in tr.steps for x in s.sources})
    print(f'{path:38} {str(tr.final):10} {chain:28} {src}')


## Against the team's first hand-typed roster

Kept as a reference, never read as an input.


In [ ]:
from egress_personas.compare import compare

print(compare(pop, Path('reference/legacy_roster_2026-09.csv')))


## The full report


In [ ]:
from IPython.display import Markdown

Markdown(markdown(pop))
